# Нейросетевые операторы (FNO)

**Проект «ИИ для учёных».** Практический блокнот к странице алгоритма «Нейросетевые операторы».

## Что делает метод

Классический решатель дифференциальных уравнений вычисляет одно решение для одного набора начальных или граничных условий. Если условия меняются — нужен новый запуск. Нейросетевой оператор обучается иначе: он усваивает **соответствие между целыми функциями** — например, между коэффициентом уравнения и его решением — и затем предсказывает решение для любого нового коэффициента за миллисекунды.

Метод особенно полезен в науке, когда один и тот же тип уравнения нужно решать тысячи раз с разными параметрами: при Монте-Карло анализе неопределённостей, при подборе обратных задач, в суррогатном моделировании.

В этом блокноте мы построим минимальный **Fourier Neural Operator (FNO)** на PyTorch: он будет учиться вычислять неопределённый интеграл произвольной функции. Это простейший пример оператора, у которого есть точный аналитический ответ для проверки. Сеть обучается на CPU за полминуты. Расчётное время прохождения блокнота — около пяти минут.

## Интуиция за методом

Обычная нейросеть — это **функция**: принимает вектор чисел, выдаёт вектор чисел. Нейросетевой оператор — это **оператор**: принимает функцию (набор значений на сетке), выдаёт другую функцию на той же или другой сетке.

Идея FNO состоит в следующем: вместо обычных свёрток в пространстве координат применяются «свёртки» в пространстве частот. Функция на сетке переводится в набор фурье-коэффициентов, коэффициенты умножаются на обучаемую матрицу (шаг в пространстве частот), затем результат переводится обратно. Этот блок называется **спектральным слоем** (SpectralConv1d).

Почему частоты? Потому что решения дифференциальных уравнений часто имеют простую структуру в частотной области — несколько доминирующих мод. Умножать на матрицу в этом пространстве — значит учить оператор, как решение реагирует на каждую частоту входных данных.

**Ключевое свойство, важное для науки:** обученный FNO применим к любому шагу сетки — разрешение можно менять без переобучения. Это называется *инвариантностью к дискретизации* и отличает нейросетевые операторы от обычных сетей.

**Связь с физически-информированными нейронными сетями (PINN):** PINN встраивает уравнение в функцию потерь и решает одно конкретное задание. FNO обучается на семействе решений и выдаёт ответ для любого входного условия без повторного обучения. Оба метода устраняют необходимость запускать численный решатель при каждом новом условии — но делают это по-разному. Для знакомства с PINN см. блокнот [physics-informed-neural-networks.ipynb](physics-informed-neural-networks.ipynb).

**Терминологический минимум:**
- **Оператор** — отображение из пространства функций в пространство функций (например, оператор дифференцирования или интегрирования).
- **БПФ** (быстрое преобразование Фурье) — алгоритм перехода от значений функции на сетке к её фурье-коэффициентам и обратно.
- **Мода** — одна гармоника (синус или косинус с определённой частотой) в разложении функции.
- **Эпоха** — один полный проход по всем обучающим примерам.

## 1. Установка библиотек

В Google Colab PyTorch и NumPy уже установлены. Ячейка ниже гарантирует нужные версии и при локальном запуске.

In [ ]:
%pip install -q torch numpy matplotlib

## 2. Единый стиль графиков

Все графики оформляются в едином визуальном стиле сайта «ИИ для учёных». Ниже встроено содержимое стилевого шаблона `notebooks/viz_style.py`.

In [ ]:
# Единый стилевой шаблон графиков проекта «ИИ для учёных».
# Значения синхронизированы с токенами темы сайта (styles/tokens.css).
VIZ_TOKENS = {
    "background": "#ffffff",
    "node_fill":  "#eef1f6",
    "node_text":  "#0e1726",
    "edge":       "#4d5e78",
    "grid":       "#dce2ec",
    "series":     ["#2563eb", "#0d9488", "#b45309", "#4d5e78"],
}
VIZ = VIZ_TOKENS


def apply_viz_style():
    """Настраивает matplotlib под единый стиль сайта."""
    import matplotlib as mpl
    from cycler import cycler
    t = VIZ_TOKENS
    mpl.rcParams.update({
        "font.family": "sans-serif",
        "font.sans-serif": ["Segoe UI", "DejaVu Sans", "Arial", "Helvetica"],
        "font.size": 12, "axes.titlesize": 15, "axes.titleweight": "semibold",
        "axes.labelsize": 13, "xtick.labelsize": 11, "ytick.labelsize": 11,
        "legend.fontsize": 11,
        "figure.facecolor": t["background"], "axes.facecolor": t["background"],
        "savefig.facecolor": t["background"], "figure.dpi": 110, "savefig.dpi": 150,
        "axes.prop_cycle": cycler(color=t["series"]),
        "text.color": t["node_text"], "axes.labelcolor": t["node_text"],
        "axes.titlecolor": t["node_text"], "axes.edgecolor": t["edge"],
        "xtick.color": t["edge"], "ytick.color": t["edge"], "axes.linewidth": 1.2,
        "axes.grid": True, "grid.color": t["grid"], "grid.linewidth": 1.0,
        "grid.linestyle": "-", "axes.axisbelow": True,
        "lines.linewidth": 2.0, "lines.markersize": 6, "patch.linewidth": 1.5,
        "axes.spines.top": False, "axes.spines.right": False,
        "legend.frameon": False,
    })


def get_palette(n=None):
    """Возвращает список цветов рядов из токенов темы."""
    series = VIZ_TOKENS["series"]
    if n is None:
        return list(series)
    return [series[i % len(series)] for i in range(n)]


apply_viz_style()
print("Единый стиль графиков подключён.")

## 3. Синтетические данные: оператор интегрирования

Задача оператора: по входной функции $a(x)$ предсказать её первообразную $u(x)$ с нулём в точке $x=0$:

$$u(x) = \int_0^x a(t)\, dt.$$

Для генерации разнообразных входных функций используем случайные гладкие суперпозиции косинусов с независимыми коэффициентами. Точный ответ считается численно (метод трапеций) — он используется **только как метка** для обучения и последующей проверки.

Такой выбор задачи удобен тем, что:
- у неё есть точное решение для любого входа;
- функции $a(x)$ легко генерировать синтетически;
- результат визуально прозрачен — проверить глазами просто.

In [ ]:
import numpy as np
import torch

# Фиксируем генераторы случайных чисел для воспроизводимости.
np.random.seed(42)
torch.manual_seed(42)

N_GRID = 64       # точек на сетке
N_TRAIN = 800     # обучающих пар (a, u)
N_TEST  = 200     # тестовых пар
N_MODES = 8       # максимальное число гармоник в a(x)

x = np.linspace(0, 1, N_GRID, dtype=np.float32)   # сетка на [0, 1]
dx = x[1] - x[0]


def generate_batch(n_samples, n_grid=N_GRID, n_modes=N_MODES):
    """Генерирует n_samples пар (a, u): случайная функция и её интеграл."""
    # Сетка строится по запрошенному n_grid (нужно для демонстрации
    # инвариантности к разрешению ниже): не опираемся на глобальные x/dx.
    grid = np.linspace(0, 1, n_grid, dtype=np.float32)
    d = grid[1] - grid[0]
    # Коэффициенты перед каждой гармоникой ~ Uniform(-1, 1)
    freqs = np.arange(1, n_modes + 1)
    coefs = np.random.uniform(-1, 1, size=(n_samples, n_modes)).astype(np.float32)
    # a(x) = сумма coef_k * cos(2*pi*k*x)
    basis = np.cos(2 * np.pi * freqs[None, :, None]
                   * grid[None, None, :]).astype(np.float32)  # (1, n_modes, n_grid)
    a = (coefs[:, :, None] * basis).sum(axis=1)            # (n_samples, n_grid)
    # Точный интеграл: методом трапеций нарастающим итогом.
    u = np.cumsum((a[:, :-1] + a[:, 1:]) / 2 * d, axis=1)
    u = np.concatenate([np.zeros((n_samples, 1), dtype=np.float32), u], axis=1)
    return a, u


a_train, u_train = generate_batch(N_TRAIN)
a_test,  u_test  = generate_batch(N_TEST)

# Переводим в тензоры PyTorch: форма (N, 1, N_GRID) — канал в средней оси.
to_tensor = lambda arr: torch.from_numpy(arr).unsqueeze(1)   # (N, 1, L)
A_train = to_tensor(a_train)
U_train = to_tensor(u_train)
A_test  = to_tensor(a_test)
U_test  = to_tensor(u_test)

print(f"Обучающих примеров: {N_TRAIN}, тестовых: {N_TEST}")
print(f"Размер сетки: {N_GRID} точек на [0, 1]")
print(f"Форма обучающего тензора A: {A_train.shape}")

## 4. Применение метода

### Шаг 1. Спектральный свёрточный слой (SpectralConv1d)

Это ключевой строительный блок FNO. Он работает в три этапа:

1. **Прямое БПФ**: переводим вход в пространство частот (`torch.fft.rfft`).
2. **Умножение на обучаемые веса в пространстве частот**: берём первые `n_modes` коэффициентов и умножаем на комплексную матрицу весов. Остальные коэффициенты (высокие частоты) обнуляются — слой низкочастотный.
3. **Обратное БПФ**: возвращаемся к исходному пространству (`torch.fft.irfft`).

Параллельно вход проходит через обычный линейный слой (проекция $W \cdot v$). Результаты складываются и пропускаются через нелинейность. Это полный FNO-блок.

In [ ]:
import torch.nn as nn


class SpectralConv1d(nn.Module):
    """Спектральный свёрточный слой: умножение в пространстве частот."""

    def __init__(self, in_ch, out_ch, n_modes):
        super().__init__()
        self.in_ch   = in_ch
        self.out_ch  = out_ch
        self.n_modes = n_modes
        # Обучаемые комплексные веса: (in_ch, out_ch, n_modes)
        scale = 1 / (in_ch * out_ch)
        self.weight = nn.Parameter(
            scale * torch.randn(in_ch, out_ch, n_modes, dtype=torch.cfloat))

    def forward(self, x):
        # x: (batch, in_ch, L)
        B, _, L = x.shape
        # 1. Прямое БПФ: (batch, in_ch, L//2+1)
        x_ft = torch.fft.rfft(x)
        # 2. Умножение первых n_modes коэффициентов на веса.
        # Результат: (batch, out_ch, n_modes)
        out_ft = torch.zeros(B, self.out_ch, L // 2 + 1,
                             dtype=torch.cfloat, device=x.device)
        # einsum: bix, iox -> box  (b=batch, i=in_ch, o=out_ch, x=freq)
        out_ft[:, :, :self.n_modes] = torch.einsum(
            "bix,iox->box",
            x_ft[:, :, :self.n_modes],
            self.weight)
        # 3. Обратное БПФ: восстанавливаем длину L
        return torch.fft.irfft(out_ft, n=L)   # (batch, out_ch, L)

### Шаг 2. Архитектура FNO

Полная сеть состоит из следующих блоков:

1. **Подъём** (`lift`): линейный слой, расширяющий входной канал до `width` каналов. Это стандартный для FNO приём — проецировать во внутреннее пространство признаков перед спектральными блоками.
2. **Спектральные блоки** (`layers`): несколько одинаковых блоков, каждый из которых объединяет спектральный слой и обычную линейную проекцию. Нелинейность `GeLU` применяется между блоками.
3. **Проекция** (`project`): линейный слой, сужающий `width` каналов обратно до 1 выходного канала.

Параметры модели намеренно малы (width=20, 3 слоя, 8 мод) для быстрого обучения на CPU.

In [ ]:
class FNO1d(nn.Module):
    """Минимальный одномерный Fourier Neural Operator."""

    def __init__(self, n_modes=8, width=20, n_layers=3):
        super().__init__()
        self.lift    = nn.Linear(1, width)    # подъём в width каналов
        self.layers  = nn.ModuleList([
            SpectralConv1d(width, width, n_modes) for _ in range(n_layers)])
        self.ws      = nn.ModuleList([
            nn.Conv1d(width, width, kernel_size=1) for _ in range(n_layers)])
        self.project = nn.Linear(width, 1)    # сужение обратно до 1 канала

    def forward(self, x):
        # x: (batch, 1, L) — входная функция на сетке
        # Подъём: (batch, 1, L) -> (batch, L, 1) -> (batch, L, width) -> (batch, width, L)
        x = self.lift(x.permute(0, 2, 1)).permute(0, 2, 1)
        for spec, w in zip(self.layers, self.ws):
            # Спектральная ветвь + локальная ветвь, затем нелинейность.
            x = torch.nn.functional.gelu(spec(x) + w(x))
        # Проекция: (batch, width, L) -> (batch, L, width) -> (batch, L, 1) -> (batch, 1, L)
        x = self.project(x.permute(0, 2, 1)).permute(0, 2, 1)
        return x  # (batch, 1, L)


model = FNO1d(n_modes=8, width=20, n_layers=3)
n_params = sum(p.numel() for p in model.parameters())
print(f"Параметров в сети: {n_params}")

### Шаг 3. Обучение

Обучаем FNO минимизировать среднеквадратичную ошибку между предсказанной первообразной и точным ответом. Для устойчивости используем относительную ошибку в норме $L^2$: она масштабируется с амплитудой функций и не зависит от числа точек сетки.

Оптимизатор Adam с расписанием шага (StepLR) позволяет за небольшое число эпох достичь хорошей точности.

In [ ]:
from torch.utils.data import DataLoader, TensorDataset

BATCH_SIZE = 64
N_EPOCHS   = 30

ds      = TensorDataset(A_train, U_train)
loader  = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=True)
opt     = torch.optim.Adam(model.parameters(), lr=1e-3)
sched   = torch.optim.lr_scheduler.StepLR(opt, step_size=10, gamma=0.5)


def rel_l2(pred, target):
    """Относительная ошибка в норме L2 по батчу."""
    diff  = (pred - target).norm(dim=-1)
    denom = target.norm(dim=-1).clamp(min=1e-8)
    return (diff / denom).mean()


history = []
for epoch in range(1, N_EPOCHS + 1):
    model.train()
    ep_loss = 0.0
    for a_b, u_b in loader:
        opt.zero_grad()
        loss = rel_l2(model(a_b), u_b)
        loss.backward()
        opt.step()
        ep_loss += loss.item() * len(a_b)
    sched.step()
    history.append(ep_loss / N_TRAIN)
    if epoch % 5 == 0:
        print(f"Эпоха {epoch:3d}: относительная L2-ошибка = {history[-1]:.4f}")

print("Обучение завершено.")

### Шаг 4. Оценка на тестовых данных и визуализация

Построим два графика:

1. **Предсказание против точного ответа** для трёх тестовых функций. Ни одну из них сеть не видела при обучении — это честная проверка обобщения оператора.
2. **Кривая сходимости** — как убывала ошибка по эпохам.

Затем демонстрируем **инвариантность к разрешению**: применяем обученную сеть к той же функции, дискретизованной на вдвое более плотной сетке (128 точек). Сеть не переобучалась на этом разрешении, но справляется — именно это отличает нейросетевой оператор от обычной регрессии.

In [ ]:
import matplotlib.pyplot as plt

model.eval()
with torch.no_grad():
    U_pred = model(A_test).squeeze(1).numpy()   # (N_TEST, N_GRID)

test_err = np.mean(
    np.linalg.norm(U_pred - u_test, axis=1)
    / np.linalg.norm(u_test, axis=1).clip(1e-8))
print(f"Средняя относительная L2-ошибка на тесте: {test_err:.4f}")

# --- График 1: предсказание против точного ответа ---
n_show = 3
colors = get_palette(4)
fig, axes = plt.subplots(1, 2, figsize=(13.5, 5.2))

for i in range(n_show):
    lw = 2.5 - i * 0.4
    axes[0].plot(x, u_test[i], color=colors[i], linewidth=lw,
                 label=f"точный ответ #{i+1}")
    axes[0].plot(x, U_pred[i], color=colors[i], linewidth=lw,
                 linestyle="--", alpha=0.85)

# Добавляем вручную легенду (сплошная = точная, пунктир = FNO)
from matplotlib.lines import Line2D
legend_items = [
    Line2D([0], [0], color=colors[i], lw=2.5 - i * 0.4,
           label=f"пример {i+1}") for i in range(n_show)
] + [
    Line2D([0], [0], color="grey", lw=1.5, linestyle="-",  label="точный ответ"),
    Line2D([0], [0], color="grey", lw=1.5, linestyle="--", label="предсказание FNO"),
]
axes[0].legend(handles=legend_items, loc="upper left", ncol=1)
axes[0].set_title("Предсказание оператора против точного ответа")
axes[0].set_xlabel("Координата x")
axes[0].set_ylabel("u(x) — первообразная")

# --- График 2: сходимость обучения ---
axes[1].plot(range(1, N_EPOCHS + 1), history, color=colors[0])
axes[1].set_title("Сходимость обучения FNO")
axes[1].set_xlabel("Эпоха")
axes[1].set_ylabel("Относительная L2-ошибка")

fig.tight_layout()
plt.show()

**Как читать эти графики.**

Левый — главный результат: сплошные линии — точные первообразные, пунктирные — предсказания FNO для трёх тестовых функций, которые сеть не видела при обучении. Хорошая модель: пунктир практически совпадает со сплошной линией. Если кривые расходятся на концах или в максимумах — это говорит о недостаточном числе мод или эпох обучения.

Правый — кривая сходимости: должна монотонно убывать. Скачки или плато после 10-й эпохи — сигнал, что шаг обучения можно снизить раньше.

### Шаг 5. Инвариантность к разрешению сетки

Обученная сеть принимала функции на сетке из 64 точек. Теперь проверим: будет ли она работать на сетке из 128 точек — то есть при вдвое большем разрешении — **без повторного обучения**?

Это возможно, потому что FNO работает через БПФ: умножаемые частоты (первые 8 мод) одни и те же вне зависимости от числа точек сетки.

In [ ]:
# Генерируем те же функции, но на сетке 128 точек.
torch.manual_seed(99)
np.random.seed(99)
x_fine = np.linspace(0, 1, 128, dtype=np.float32)
dx_fine = x_fine[1] - x_fine[0]

a_fine, u_fine = generate_batch(5, n_grid=128, n_modes=N_MODES)
A_fine = torch.from_numpy(a_fine).unsqueeze(1)   # (5, 1, 128)

model.eval()
with torch.no_grad():
    U_fine_pred = model(A_fine).squeeze(1).numpy()   # (5, 128)

err_fine = np.mean(
    np.linalg.norm(U_fine_pred - u_fine, axis=1)
    / np.linalg.norm(u_fine, axis=1).clip(1e-8))

fig, ax = plt.subplots(figsize=(9.5, 5.0))
colors = get_palette(4)
for i in range(3):
    lw = 2.5 - i * 0.4
    ax.plot(x_fine, u_fine[i], color=colors[i], linewidth=lw)
    ax.plot(x_fine, U_fine_pred[i], color=colors[i], linewidth=lw,
            linestyle="--", alpha=0.85)

from matplotlib.lines import Line2D
ax.legend(handles=[
    Line2D([0], [0], color="grey", lw=2, linestyle="-",  label="точный ответ (128 точек)"),
    Line2D([0], [0], color="grey", lw=2, linestyle="--", label="FNO, обученный на 64 точках"),
], loc="upper left")
ax.set_title("Инвариантность к разрешению: FNO на удвоенной сетке")
ax.set_xlabel("Координата x")
ax.set_ylabel("u(x)")
fig.tight_layout()
plt.show()

print(f"Средняя относительная L2-ошибка на сетке 128 точек: {err_fine:.4f}")
print(f"(Обучение проходило на сетке 64 точки — разрешение незнакомое)")

**Как читать этот график.**

Сеть обучалась на 64 точках, а здесь работает на 128. Если пунктир и сплошная линия по-прежнему близки — инвариантность к разрешению подтверждена. Сравните ошибку на 64 точках (из предыдущей ячейки) с ошибкой на 128: небольшой рост допустим. Существенный рост означает, что выбранные 8 мод не охватывают высокочастотную структуру входных функций.

## 5. Интерпретация результата

- **Оператор, а не функция**: обычная нейросеть обучается давать ответ для фиксированного числа входов. FNO обучается на семействе функций и применяется к любой новой функции на любой сетке — это принципиальное отличие.
- **Зачем это учёному**: одна обучающая выборка (800 пар уравнение/решение) заменяет 800 запусков численного решателя. Для каждого нового входа FNO даёт ответ за миллисекунды против секунд или минут у решателя. Выигрыш растёт с размерностью задачи.
- **Число мод** (`n_modes`) задаёт верхнюю частоту, которую оператор отслеживает. Чем больше мод — тем лучше высокочастотные детали, но дольше обучение и сложнее обобщение.
- **Ширина сети** (`width`) — внутренняя размерность признаков. Больше ширина — больше выразительность, но и число параметров.
- **Относительная L2-ошибка** ниже 5% обычно означает практически полезное приближение. Для конкретного уравнения следует сравнить с погрешностью численного решателя.
- **Ограничения**: FNO предполагает регулярную сетку; для нерегулярных сеток (конечные элементы) нужны геометрически-осведомлённые операторы (Geo-FNO или Graph Neural Operator).

## 6. Подставьте свои данные

Чтобы применить FNO к собственному оператору, нужно:

1. Подготовить датасет пар `(a, u)`: каждый вход и выход — функция на одинаковой регулярной сетке из `N_GRID` точек.
2. Загрузить массивы формы `(N_SAMPLES, N_GRID)` и преобразовать в тензоры формы `(N_SAMPLES, 1, N_GRID)`.
3. При необходимости изменить `N_GRID`, `N_MODES`, `width` в конструкторе.

Для многомерных полей (2D уравнения) понадобится `SpectralConv2d` — двумерный аналог, принцип тот же.

In [ ]:
# --- Шаблон загрузки своих данных ---
# Предполагается, что у вас есть:
#   a_my: numpy-массив формы (N, N_GRID) — входные функции
#   u_my: numpy-массив формы (N, N_GRID) — соответствующие решения/выходы

# import numpy as np
# import torch
#
# a_my = np.load('inputs.npy').astype('float32')   # (N, N_GRID)
# u_my = np.load('outputs.npy').astype('float32')  # (N, N_GRID)
#
# A_my = torch.from_numpy(a_my).unsqueeze(1)  # (N, 1, N_GRID)
# U_my = torch.from_numpy(u_my).unsqueeze(1)  # (N, 1, N_GRID)
#
# # Создайте модель с нужной размерностью:
# # my_model = FNO1d(n_modes=16, width=32, n_layers=4)
#
# # Далее повторите ячейки раздела 4 (обучение и оценка).

## 7. Поэкспериментируйте сами

Измените один параметр, перезапустите разделы 3–4 и сравните результат.

| Параметр | Что изменить | Что наблюдать |
|---|---|---|
| `n_modes` | Уменьшить до 4 | Ошибка растёт — FNO пропускает высокочастотные детали |
| `n_modes` | Увеличить до 16 | Ошибка падает, но обучение медленнее |
| `width` | Увеличить до 40 | Больше параметров, может дать точнее — или переобучиться |
| `N_EPOCHS` | Увеличить до 60 | Дополнительно снижает ошибку после расписания шага |
| Задача | Вместо интеграла — производная (`np.gradient`) | Более трудная задача: ошибка будет выше; попробуйте больше мод |

Обратите внимание: FNO чувствителен к масштабу входных функций. Если ваши данные сильно разнятся по амплитуде, нормализация входов помогает обучению.

## Краткие выводы

- Нейросетевые операторы обучают **отображение между функциями**, а не между векторами: входная функция $a(x)$ → выходная функция $u(x)$.
- FNO реализует это через умножение в пространстве частот (БПФ): обучаемые веса определяют, как каждая частота входа влияет на выход.
- Ключевое преимущество — **инвариантность к разрешению**: модель, обученная на одной сетке, работает на любой другой сетке без переобучения.
- Главная практическая польза: одна обучающая выборка заменяет тысячи запусков численного решателя при параметрических исследованиях.
- Ограничения метода: требует регулярной сетки; для нерегулярной геометрии или трёхмерных задач нужны расширения архитектуры.

## Три вопроса для самопроверки

Ответьте до того, как раскроете подсказку, — это проверяет, что метод понят, а не просто запущен.

1. Вы обучили FNO на функциях с 8 модами и применяете его к входным функциям, содержащим значимые компоненты на 15-й и 20-й гармониках. Как это отразится на качестве предсказания и почему — с точки зрения архитектуры спектрального слоя?
2. Коллега предлагает решить ту же задачу с помощью физически-информированной нейронной сети (PINN). В каких двух ситуациях FNO предпочтительнее, а в какой PINN имеет принципиальное преимущество перед FNO?
3. Ошибка FNO на сетке 64 точки составила 3%, а на сетке 128 точек — 11% при том же обученном операторе. Назовите наиболее вероятную причину ухудшения и как её проверить.

<details>
<summary>Показать ориентиры для ответов</summary>

1. Спектральный слой умножает только первые `n_modes=8` фурье-коэффициентов на обучаемые веса; коэффициенты с номером выше 8 обнуляются. Высокочастотные компоненты (гармоники 15 и 20) будут полностью «срезаны», и их вклад в выходную функцию потеряется. Результат: выход FNO воспроизведёт низкочастотный скелет, но пропустит тонкую структуру. Решение — увеличить `n_modes`.
2. FNO предпочтительнее, когда: (а) одно и то же уравнение нужно решать тысячи раз с разными начальными/граничными условиями или параметрами (параметрические исследования, Монте-Карло); (б) нужна скорость вывода в реальном времени после однократного обучения. PINN имеет преимущество, когда обучающих пар «вход–решение» нет совсем и доступно только уравнение с конкретными условиями — FNO без обучающих данных обучить невозможно.
3. Наиболее вероятная причина: входные функции на сетке 128 точек содержат значимые высокочастотные компоненты (гармоники выше 32), которых нет у 64-точечных функций и которые FNO не обрабатывает. Проверка: посмотреть спектр нескольких тестовых функций на 128 точках — если за 32-й гармоникой есть заметная энергия, нужно либо увеличить `n_modes`, либо применить сглаживание перед подачей в сеть.
</details>